# Notebook 2 Final — Descriptor/Fingerprint Results with Entity-Disjoint Splits

This notebook consolidates the final classical-machine-learning results for the dissertation analysis of solvation Gibbs free energy prediction.

It combines:

- the original fixed random row split from Notebook 2,
- the new scaffold-disjoint split,
- the new leave-solvents-out split,
- the new leave-solutes-out split,
- and the earlier solvent-specific water hold-out test.

The goal is not to rerun model training here. This is a result-audit and comparison notebook built from the completed Notebook 2 runs.

## Input Result Audit

The uploaded split-result files were checked before this notebook was created:

- `section8_final_test_results_scaffold_disjoint.csv`: 14 model rows, expected Section 8 metrics present.
- `section8_final_test_results_leave_solvents_out.csv`: 13 model rows, expected Section 8 metrics present.
- `section8_final_test_results_leave_solutes_out.csv`: 14 model rows, expected Section 8 metrics present.
- The three `run_summary_*.json` files match the notebook outputs for split size and best model.

The original fixed-random results were extracted from the executed `note2_fixed.ipynb` Section 8 output.

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.float_format", lambda v: f"{v:.4f}")

In [ ]:
RESULT_TABLES = json.loads(r"""{
  "fixed_random": [
    {
      "Features": "Descriptors",
      "Model": "SVR-rbf",
      "Validation MAE": 0.2295,
      "Validation RMSE": 0.5176,
      "Test R\u00b2": 0.9513,
      "Test RMSE": 0.4967,
      "Test MAE": 0.2113,
      "Train+Val R\u00b2": 0.9964,
      "Time (s)": 11.5
    },
    {
      "Features": "Descriptors",
      "Model": "RandomForest",
      "Validation MAE": 0.2816,
      "Validation RMSE": 0.5546,
      "Test R\u00b2": 0.9632,
      "Test RMSE": 0.4317,
      "Test MAE": 0.2442,
      "Train+Val R\u00b2": 0.9925,
      "Time (s)": 6.8
    },
    {
      "Features": "Descriptors",
      "Model": "GradientBoosting",
      "Validation MAE": 0.3085,
      "Validation RMSE": 0.5291,
      "Test R\u00b2": 0.9642,
      "Test RMSE": 0.4261,
      "Test MAE": 0.2862,
      "Train+Val R\u00b2": 0.9831,
      "Time (s)": 29.8
    },
    {
      "Features": "Fingerprints",
      "Model": "RandomForest",
      "Validation MAE": 0.521,
      "Validation RMSE": 1.0863,
      "Test R\u00b2": 0.8566,
      "Test RMSE": 0.8526,
      "Test MAE": 0.4692,
      "Train+Val R\u00b2": 0.9558,
      "Time (s)": 13.0
    },
    {
      "Features": "Descriptors",
      "Model": "LinearSVR",
      "Validation MAE": 0.6177,
      "Validation RMSE": 1.0914,
      "Test R\u00b2": 0.8358,
      "Test RMSE": 0.9123,
      "Test MAE": 0.5887,
      "Train+Val R\u00b2": 0.8119,
      "Time (s)": 12.7
    },
    {
      "Features": "Descriptors",
      "Model": "Ridge",
      "Validation MAE": 0.6402,
      "Validation RMSE": 0.95,
      "Test R\u00b2": 0.8328,
      "Test RMSE": 0.9206,
      "Test MAE": 0.6509,
      "Train+Val R\u00b2": 0.8374,
      "Time (s)": 0.3
    },
    {
      "Features": "Descriptors",
      "Model": "Linear",
      "Validation MAE": 0.6279,
      "Validation RMSE": 0.9332,
      "Test R\u00b2": 0.7855,
      "Test RMSE": 1.0428,
      "Test MAE": 0.6554,
      "Train+Val R\u00b2": 0.8413,
      "Time (s)": 0.5
    },
    {
      "Features": "Descriptors",
      "Model": "SGD",
      "Validation MAE": 0.6676,
      "Validation RMSE": 0.9825,
      "Test R\u00b2": 0.8208,
      "Test RMSE": 0.9531,
      "Test MAE": 0.6806,
      "Train+Val R\u00b2": 0.8271,
      "Time (s)": 0.5
    },
    {
      "Features": "Fingerprints",
      "Model": "LinearSVR",
      "Validation MAE": 0.722,
      "Validation RMSE": 1.4227,
      "Test R\u00b2": 0.7235,
      "Test RMSE": 1.1839,
      "Test MAE": 0.6828,
      "Train+Val R\u00b2": 0.8663,
      "Time (s)": 6.0
    },
    {
      "Features": "Fingerprints",
      "Model": "Ridge",
      "Validation MAE": 0.759,
      "Validation RMSE": 1.4702,
      "Test R\u00b2": 0.75,
      "Test RMSE": 1.1258,
      "Test MAE": 0.6914,
      "Train+Val R\u00b2": 0.8802,
      "Time (s)": 1.9
    },
    {
      "Features": "Fingerprints",
      "Model": "Lasso",
      "Validation MAE": 0.6565,
      "Validation RMSE": 1.1062,
      "Test R\u00b2": 0.7562,
      "Test RMSE": 1.1117,
      "Test MAE": 0.692,
      "Train+Val R\u00b2": 0.8561,
      "Time (s)": 2.6
    },
    {
      "Features": "Fingerprints",
      "Model": "Linear",
      "Validation MAE": 0.7832,
      "Validation RMSE": 1.5477,
      "Test R\u00b2": 0.7449,
      "Test RMSE": 1.1373,
      "Test MAE": 0.6972,
      "Train+Val R\u00b2": 0.8803,
      "Time (s)": 27.3
    },
    {
      "Features": "Descriptors",
      "Model": "Lasso",
      "Validation MAE": 0.6987,
      "Validation RMSE": 1.0643,
      "Test R\u00b2": 0.7958,
      "Test RMSE": 1.0174,
      "Test MAE": 0.715,
      "Train+Val R\u00b2": 0.8036,
      "Time (s)": 1.4
    },
    {
      "Features": "Fingerprints",
      "Model": "GradientBoosting",
      "Validation MAE": 0.9752,
      "Validation RMSE": 1.4389,
      "Test R\u00b2": 0.6334,
      "Test RMSE": 1.3633,
      "Test MAE": 0.9654,
      "Train+Val R\u00b2": 0.694,
      "Time (s)": 99.6
    }
  ],
  "scaffold_disjoint": [
    {
      "Features": "Descriptors",
      "Model": "SVR-rbf",
      "Validation MAE": 0.4267178785232615,
      "Validation RMSE": 0.6767516561632492,
      "Test R\u00b2": 0.8390401107051927,
      "Test RMSE": 0.761210305544254,
      "Test MAE": 0.3228336181426491,
      "Train+Val R\u00b2": 0.997013315095988,
      "Time (s)": 11.0
    },
    {
      "Features": "Descriptors",
      "Model": "GradientBoosting",
      "Validation MAE": 0.4178021032898006,
      "Validation RMSE": 0.6415767647766518,
      "Test R\u00b2": 0.8950146398864491,
      "Test RMSE": 0.6147665060698475,
      "Test MAE": 0.3707188862325413,
      "Train+Val R\u00b2": 0.98398249341303,
      "Time (s)": 29.6
    },
    {
      "Features": "Descriptors",
      "Model": "RandomForest",
      "Validation MAE": 0.4931660710049347,
      "Validation RMSE": 0.7654407269815302,
      "Test R\u00b2": 0.8803681670249378,
      "Test RMSE": 0.6562498323016357,
      "Test MAE": 0.3863903152780032,
      "Train+Val R\u00b2": 0.9932129156351543,
      "Time (s)": 6.7
    },
    {
      "Features": "Descriptors",
      "Model": "LinearSVR",
      "Validation MAE": 0.847438518101425,
      "Validation RMSE": 1.2648498623544002,
      "Test R\u00b2": 0.7304090678563442,
      "Test RMSE": 0.9851411656758908,
      "Test MAE": 0.5983130224081085,
      "Train+Val R\u00b2": 0.8167923074967963,
      "Time (s)": 11.0
    },
    {
      "Features": "Descriptors",
      "Model": "Lasso",
      "Validation MAE": 0.746397408236522,
      "Validation RMSE": 1.076050580895516,
      "Test R\u00b2": 0.7327028977436468,
      "Test RMSE": 0.9809411475503276,
      "Test MAE": 0.6223627141012467,
      "Train+Val R\u00b2": 0.8058294168037158,
      "Time (s)": 1.8
    },
    {
      "Features": "Descriptors",
      "Model": "Linear",
      "Validation MAE": 1.0580294668279822,
      "Validation RMSE": 1.615296858602542,
      "Test R\u00b2": 0.7250580502288497,
      "Test RMSE": 0.9948699908272276,
      "Test MAE": 0.6407172512681368,
      "Train+Val R\u00b2": 0.8429570155488865,
      "Time (s)": 0.4
    },
    {
      "Features": "Descriptors",
      "Model": "Ridge",
      "Validation MAE": 0.9323470994219892,
      "Validation RMSE": 1.2722009526810354,
      "Test R\u00b2": 0.7100486790077812,
      "Test RMSE": 1.0216646639307454,
      "Test MAE": 0.6665942995778229,
      "Train+Val R\u00b2": 0.8398386334584287,
      "Time (s)": 0.3
    },
    {
      "Features": "Descriptors",
      "Model": "SGD",
      "Validation MAE": 0.8219532094285645,
      "Validation RMSE": 1.181485029276568,
      "Test R\u00b2": 0.6869651221644676,
      "Test RMSE": 1.061554244811041,
      "Test MAE": 0.7071829550850988,
      "Train+Val R\u00b2": 0.8303583148084495,
      "Time (s)": 0.5
    },
    {
      "Features": "Fingerprints",
      "Model": "RandomForest",
      "Validation MAE": 1.2209713948169396,
      "Validation RMSE": 1.7244578805154989,
      "Test R\u00b2": 0.5069254919504877,
      "Test RMSE": 1.3323002273354725,
      "Test MAE": 0.8510523798989589,
      "Train+Val R\u00b2": 0.9608152666902824,
      "Time (s)": 11.8
    },
    {
      "Features": "Fingerprints",
      "Model": "Lasso",
      "Validation MAE": 1.4971770374262805,
      "Validation RMSE": 2.5095665728420924,
      "Test R\u00b2": 0.4448483950795963,
      "Test RMSE": 1.4136816696234682,
      "Test MAE": 0.869052501684365,
      "Train+Val R\u00b2": 0.8597885954723737,
      "Time (s)": 11.3
    },
    {
      "Features": "Fingerprints",
      "Model": "GradientBoosting",
      "Validation MAE": 1.2507697579348906,
      "Validation RMSE": 1.7871147291312224,
      "Test R\u00b2": 0.3834275528301792,
      "Test RMSE": 1.4898340084279096,
      "Test MAE": 0.9978242602182232,
      "Train+Val R\u00b2": 0.695649772022102,
      "Time (s)": 134.9
    },
    {
      "Features": "Fingerprints",
      "Model": "LinearSVR",
      "Validation MAE": 2.073813226668132,
      "Validation RMSE": 2.789043078077456,
      "Test R\u00b2": -0.5815100756965417,
      "Test RMSE": 2.386062462736165,
      "Test MAE": 1.46764119365699,
      "Train+Val R\u00b2": 0.8697725888760511,
      "Time (s)": 17.6
    },
    {
      "Features": "Fingerprints",
      "Model": "Ridge",
      "Validation MAE": 2.1001392236567487,
      "Validation RMSE": 2.923826205840361,
      "Test R\u00b2": -0.6507442659489815,
      "Test RMSE": 2.437730693906977,
      "Test MAE": 1.6432639947096923,
      "Train+Val R\u00b2": 0.8825277064801844,
      "Time (s)": 2.9
    },
    {
      "Features": "Fingerprints",
      "Model": "Linear",
      "Validation MAE": 2.2240474187352923,
      "Validation RMSE": 3.110091906012276,
      "Test R\u00b2": -0.8646616610874771,
      "Test RMSE": 2.5908713447053278,
      "Test MAE": 1.8170783400430817,
      "Train+Val R\u00b2": 0.8825634500373779,
      "Time (s)": 47.5
    }
  ],
  "leave_solvents_out": [
    {
      "Features": "Descriptors",
      "Model": "RandomForest",
      "Validation MAE": 0.1971112474118319,
      "Validation RMSE": 0.3612525029131633,
      "Test R\u00b2": 0.961656637376078,
      "Test RMSE": 0.3564161672958061,
      "Test MAE": 0.2534279149421983,
      "Train+Val R\u00b2": 0.9930001225381704,
      "Time (s)": 6.9
    },
    {
      "Features": "Descriptors",
      "Model": "GradientBoosting",
      "Validation MAE": 0.267876821349487,
      "Validation RMSE": 0.3936213259082358,
      "Test R\u00b2": 0.9583563826301176,
      "Test RMSE": 0.3714381614851101,
      "Test MAE": 0.2781280158704111,
      "Train+Val R\u00b2": 0.9836321132995834,
      "Time (s)": 30.9
    },
    {
      "Features": "Descriptors",
      "Model": "SVR-rbf",
      "Validation MAE": 0.2061136452454036,
      "Validation RMSE": 0.3570381130560556,
      "Test R\u00b2": 0.922879762651034,
      "Test RMSE": 0.5054712698628485,
      "Test MAE": 0.3283591817329472,
      "Train+Val R\u00b2": 0.99648858042659,
      "Time (s)": 11.4
    },
    {
      "Features": "Fingerprints",
      "Model": "RandomForest",
      "Validation MAE": 0.3976457457063025,
      "Validation RMSE": 0.9788192372878972,
      "Test R\u00b2": 0.8318642786486317,
      "Test RMSE": 0.7463495804927442,
      "Test MAE": 0.473462225565534,
      "Train+Val R\u00b2": 0.9562430673534006,
      "Time (s)": 26.1
    },
    {
      "Features": "Fingerprints",
      "Model": "LinearSVR",
      "Validation MAE": 1.0133022513829395,
      "Validation RMSE": 1.4890151561652822,
      "Test R\u00b2": 0.7541009120694016,
      "Test RMSE": 0.9025904960697382,
      "Test MAE": 0.6071843386357266,
      "Train+Val R\u00b2": 0.8628156556001264,
      "Time (s)": 6.2
    },
    {
      "Features": "Fingerprints",
      "Model": "Ridge",
      "Validation MAE": 0.6762428007947835,
      "Validation RMSE": 1.1820250678989708,
      "Test R\u00b2": 0.7446747622836246,
      "Test RMSE": 0.9197274949526708,
      "Test MAE": 0.6179242182461238,
      "Train+Val R\u00b2": 0.8762769582185629,
      "Time (s)": 3.8
    },
    {
      "Features": "Fingerprints",
      "Model": "Linear",
      "Validation MAE": 0.6673300969016364,
      "Validation RMSE": 1.179254782075388,
      "Test R\u00b2": 0.7310241798777339,
      "Test RMSE": 0.9439933121651048,
      "Test MAE": 0.6272272125607684,
      "Train+Val R\u00b2": 0.876316148910617,
      "Time (s)": 76.8
    },
    {
      "Features": "Fingerprints",
      "Model": "Lasso",
      "Validation MAE": 0.5427635952506321,
      "Validation RMSE": 1.0502466922158047,
      "Test R\u00b2": 0.7209759035320692,
      "Test RMSE": 0.9614642765453428,
      "Test MAE": 0.6568483079345611,
      "Train+Val R\u00b2": 0.8519510668589552,
      "Time (s)": 3.3
    },
    {
      "Features": "Descriptors",
      "Model": "LinearSVR",
      "Validation MAE": 0.4695746260517262,
      "Validation RMSE": 0.7345802066382999,
      "Test R\u00b2": 0.7592283690560221,
      "Test RMSE": 0.8931305699871197,
      "Test MAE": 0.662264540890584,
      "Train+Val R\u00b2": 0.8151130194868967,
      "Time (s)": 12.6
    },
    {
      "Features": "Descriptors",
      "Model": "Lasso",
      "Validation MAE": 0.5310115029382434,
      "Validation RMSE": 0.8162720725080412,
      "Test R\u00b2": 0.7585241950094499,
      "Test RMSE": 0.8944356660095049,
      "Test MAE": 0.7076324168960993,
      "Train+Val R\u00b2": 0.80384368604,
      "Time (s)": 2.8
    },
    {
      "Features": "Descriptors",
      "Model": "Ridge",
      "Validation MAE": 0.5157486837609452,
      "Validation RMSE": 0.7793145224896927,
      "Test R\u00b2": 0.7482672813741205,
      "Test RMSE": 0.9132341225876016,
      "Test MAE": 0.7174955611602616,
      "Train+Val R\u00b2": 0.8371992434091371,
      "Time (s)": 0.4
    },
    {
      "Features": "Descriptors",
      "Model": "SGD",
      "Validation MAE": 0.5470891169189326,
      "Validation RMSE": 0.806978616487623,
      "Test R\u00b2": 0.7270290891130404,
      "Test RMSE": 0.95097802570993,
      "Test MAE": 0.7555142325681417,
      "Train+Val R\u00b2": 0.8264198074638998,
      "Time (s)": 0.6
    },
    {
      "Features": "Fingerprints",
      "Model": "GradientBoosting",
      "Validation MAE": 0.885109193106787,
      "Validation RMSE": 1.3221409468794636,
      "Test R\u00b2": 0.6448779197072688,
      "Test RMSE": 1.084678758513983,
      "Test MAE": 0.8716721413531704,
      "Train+Val R\u00b2": 0.687148057649352,
      "Time (s)": 106.3
    }
  ],
  "leave_solutes_out": [
    {
      "Features": "Descriptors",
      "Model": "SVR-rbf",
      "Validation MAE": 0.307975245763484,
      "Validation RMSE": 0.6068730253236476,
      "Test R\u00b2": 0.9179370931366309,
      "Test RMSE": 0.4815214639375437,
      "Test MAE": 0.2913825044214985,
      "Train+Val R\u00b2": 0.9965069788575028,
      "Time (s)": 10.2
    },
    {
      "Features": "Descriptors",
      "Model": "RandomForest",
      "Validation MAE": 0.6322625635378778,
      "Validation RMSE": 1.0294645286826756,
      "Test R\u00b2": 0.8518271715193703,
      "Test RMSE": 0.6470330846731741,
      "Test MAE": 0.4394842449608886,
      "Train+Val R\u00b2": 0.9931483539616484,
      "Time (s)": 6.9
    },
    {
      "Features": "Descriptors",
      "Model": "GradientBoosting",
      "Validation MAE": 0.5705259152981741,
      "Validation RMSE": 0.8192767094585705,
      "Test R\u00b2": 0.8381266526656556,
      "Test RMSE": 0.6762851887379958,
      "Test MAE": 0.4756884404262131,
      "Train+Val R\u00b2": 0.983848795471022,
      "Time (s)": 28.5
    },
    {
      "Features": "Descriptors",
      "Model": "SGD",
      "Validation MAE": 0.6830548370126369,
      "Validation RMSE": 0.9570098885632914,
      "Test R\u00b2": 0.6982444410706499,
      "Test RMSE": 0.92335741695859,
      "Test MAE": 0.6395418649232074,
      "Train+Val R\u00b2": 0.8332609307734516,
      "Time (s)": 0.5
    },
    {
      "Features": "Descriptors",
      "Model": "Ridge",
      "Validation MAE": 0.6902275352861815,
      "Validation RMSE": 0.9578220774129006,
      "Test R\u00b2": 0.6969616016912479,
      "Test RMSE": 0.9253180486134556,
      "Test MAE": 0.6405044930826324,
      "Train+Val R\u00b2": 0.8439115947665231,
      "Time (s)": 0.3
    },
    {
      "Features": "Descriptors",
      "Model": "Linear",
      "Validation MAE": 0.6777479533488647,
      "Validation RMSE": 0.9358873044045538,
      "Test R\u00b2": 0.6881209801965387,
      "Test RMSE": 0.9387182962958496,
      "Test MAE": 0.6463359071805365,
      "Train+Val R\u00b2": 0.847498181904491,
      "Time (s)": 0.5
    },
    {
      "Features": "Descriptors",
      "Model": "LinearSVR",
      "Validation MAE": 0.6737150922306447,
      "Validation RMSE": 1.0100728962665215,
      "Test R\u00b2": 0.6564259378129793,
      "Test RMSE": 0.9852634840506848,
      "Test MAE": 0.6500246730071615,
      "Train+Val R\u00b2": 0.8216009138693284,
      "Time (s)": 10.9
    },
    {
      "Features": "Descriptors",
      "Model": "Lasso",
      "Validation MAE": 0.7104962588709454,
      "Validation RMSE": 1.0011758941547777,
      "Test R\u00b2": 0.681299574221319,
      "Test RMSE": 0.9489285738888606,
      "Test MAE": 0.6741330246296876,
      "Train+Val R\u00b2": 0.8095725468797152,
      "Time (s)": 1.5
    },
    {
      "Features": "Fingerprints",
      "Model": "RandomForest",
      "Validation MAE": 1.0073104353337778,
      "Validation RMSE": 1.5036795098382494,
      "Test R\u00b2": 0.613503685524482,
      "Test RMSE": 1.044996608186129,
      "Test MAE": 0.7492048963858644,
      "Train+Val R\u00b2": 0.9582903847821136,
      "Time (s)": 11.7
    },
    {
      "Features": "Fingerprints",
      "Model": "Lasso",
      "Validation MAE": 0.9203271396793916,
      "Validation RMSE": 1.458289708404462,
      "Test R\u00b2": 0.3972412470050486,
      "Test RMSE": 1.3050103774949409,
      "Test MAE": 0.9196580695472176,
      "Train+Val R\u00b2": 0.8590995358435303,
      "Time (s)": 2.0
    },
    {
      "Features": "Fingerprints",
      "Model": "GradientBoosting",
      "Validation MAE": 1.1390791101496829,
      "Validation RMSE": 1.650975166793107,
      "Test R\u00b2": 0.4368729106171679,
      "Test RMSE": 1.2613784650375828,
      "Test MAE": 0.9452499414715008,
      "Train+Val R\u00b2": 0.6994334407704548,
      "Time (s)": 99.1
    },
    {
      "Features": "Fingerprints",
      "Model": "LinearSVR",
      "Validation MAE": 1.3197279300011258,
      "Validation RMSE": 1.9746524856530072,
      "Test R\u00b2": -0.3658160423498469,
      "Test RMSE": 1.9644379121225397,
      "Test MAE": 1.5004893318136383,
      "Train+Val R\u00b2": 0.8679050402716793,
      "Time (s)": 5.6
    },
    {
      "Features": "Fingerprints",
      "Model": "Ridge",
      "Validation MAE": 1.3826399609327973,
      "Validation RMSE": 2.0382644409428377,
      "Test R\u00b2": -0.3924640468717855,
      "Test RMSE": 1.9835091016788204,
      "Test MAE": 1.5144000426602706,
      "Train+Val R\u00b2": 0.8814917877771562,
      "Time (s)": 2.1
    },
    {
      "Features": "Fingerprints",
      "Model": "Linear",
      "Validation MAE": 1.6327330397100883,
      "Validation RMSE": 2.325004504976883,
      "Test R\u00b2": -0.6277488475126207,
      "Test RMSE": 2.144548639004499,
      "Test MAE": 1.6213423011078252,
      "Train+Val R\u00b2": 0.8815404360982255,
      "Time (s)": 32.1
    }
  ]
}""")
RUN_SUMMARIES = json.loads(r"""{
  "fixed_random": {
    "split_name": "fixed_random",
    "n_train": 4991,
    "n_validation": 624,
    "n_test": 624,
    "best_model": "SVR-rbf",
    "best_features": "Descriptors",
    "best_test_mae": 0.2113,
    "best_test_rmse": 0.4967,
    "best_test_r2": 0.9513
  },
  "scaffold_disjoint": {
    "split_name": "scaffold_disjoint",
    "n_train": 5036,
    "n_validation": 833,
    "n_test": 370,
    "best_model": "SVR-rbf",
    "best_features": "Descriptors",
    "best_test_mae": 0.3228336181426491,
    "best_test_rmse": 0.761210305544254,
    "best_test_r2": 0.8390401107051927
  },
  "leave_solvents_out": {
    "split_name": "leave_solvents_out",
    "n_train": 5335,
    "n_validation": 413,
    "n_test": 491,
    "best_model": "RandomForest",
    "best_features": "Descriptors",
    "best_test_mae": 0.2534279149421983,
    "best_test_rmse": 0.35641616729580616,
    "best_test_r2": 0.9616566373760781
  },
  "leave_solutes_out": {
    "split_name": "leave_solutes_out",
    "n_train": 4993,
    "n_validation": 617,
    "n_test": 629,
    "best_model": "SVR-rbf",
    "best_features": "Descriptors",
    "best_test_mae": 0.29138250442149854,
    "best_test_rmse": 0.4815214639375437,
    "best_test_r2": 0.9179370931366309
  }
}""")
WATER_HOLDOUT_ROWS = json.loads(r"""[
  {
    "Solvent": "Water",
    "SMILES": "O",
    "n_train": 5597,
    "n_test": 642,
    "R\u00b2": 0.0306,
    "MAE": 2.8091,
    "RMSE": 3.7855
  },
  {
    "Solvent": "DMSO",
    "SMILES": "CS(C)=O",
    "n_train": 6222,
    "n_test": 17,
    "R\u00b2": 0.25,
    "MAE": 1.0159,
    "RMSE": 1.1254
  },
  {
    "Solvent": "THF",
    "SMILES": "C1CCOC1",
    "n_train": 6172,
    "n_test": 67,
    "R\u00b2": 0.9488,
    "MAE": 0.358,
    "RMSE": 0.464
  },
  {
    "Solvent": "DCM",
    "SMILES": "ClCCl",
    "n_train": 6206,
    "n_test": 33,
    "R\u00b2": 0.9303,
    "MAE": 0.3157,
    "RMSE": 0.5928
  },
  {
    "Solvent": "Octanol",
    "SMILES": "CCCCCCCCO",
    "n_train": 6110,
    "n_test": 129,
    "R\u00b2": 0.9062,
    "MAE": 0.2902,
    "RMSE": 0.7348
  },
  {
    "Solvent": "Hexadecane",
    "SMILES": "CCCCCCCCCCCCCCCC",
    "n_train": 5960,
    "n_test": 279,
    "R\u00b2": 0.9369,
    "MAE": 0.2573,
    "RMSE": 0.3832
  },
  {
    "Solvent": "Acetone",
    "SMILES": "CC(C)=O",
    "n_train": 6162,
    "n_test": 77,
    "R\u00b2": 0.9652,
    "MAE": 0.2297,
    "RMSE": 0.3476
  },
  {
    "Solvent": "Ethanol",
    "SMILES": "CCO",
    "n_train": 6174,
    "n_test": 65,
    "R\u00b2": 0.9835,
    "MAE": 0.2177,
    "RMSE": 0.3287
  },
  {
    "Solvent": "Toluene",
    "SMILES": "Cc1ccccc1",
    "n_train": 6128,
    "n_test": 111,
    "R\u00b2": 0.9761,
    "MAE": 0.2,
    "RMSE": 0.2922
  }
]""")

result_tables = {name: pd.DataFrame(rows) for name, rows in RESULT_TABLES.items()}
run_summaries = RUN_SUMMARIES
water_holdout = pd.DataFrame(WATER_HOLDOUT_ROWS)

for name, table in result_tables.items():
    print(name, table.shape)
    assert {"Features", "Model", "Test MAE", "Test RMSE", "Test R²"}.issubset(table.columns)

print("All embedded result tables have the expected metric columns.")

## Headline Comparison

| Protocol | Train | Validation | Test | Best model | Test MAE | Test RMSE | Test R² |
| --- | --- | --- | --- | --- | --- | --- | --- |
| Fixed random row split | 4991 | 624 | 624 | SVR-rbf (Descriptors) | 0.2113 | 0.4967 | 0.9513 |
| Scaffold-disjoint | 5036 | 833 | 370 | SVR-rbf (Descriptors) | 0.3228 | 0.7612 | 0.8390 |
| Leave-solvents-out | 5335 | 413 | 491 | RandomForest (Descriptors) | 0.2534 | 0.3564 | 0.9617 |
| Leave-solutes-out | 4993 | 617 | 629 | SVR-rbf (Descriptors) | 0.2914 | 0.4815 | 0.9179 |

In [ ]:
# Build the headline comparison table programmatically.
protocol_labels = {
    "fixed_random": "Fixed random row split",
    "scaffold_disjoint": "Scaffold-disjoint",
    "leave_solvents_out": "Leave-solvents-out",
    "leave_solutes_out": "Leave-solutes-out",
}

comparison_rows = []

for split_name, table in result_tables.items():
    summary = run_summaries[split_name]
    best = table.sort_values("Test MAE").iloc[0]

    comparison_rows.append({
        "Protocol": protocol_labels[split_name],
        "Entity constraint": {
            "fixed_random": "None; row-level interpolation",
            "scaffold_disjoint": "No solute-scaffold overlap",
            "leave_solvents_out": "No solvent overlap",
            "leave_solutes_out": "No solute overlap",
        }[split_name],
        "Train": int(summary["n_train"]),
        "Validation": int(summary["n_validation"]),
        "Test": int(summary["n_test"]),
        "Best model": best["Model"],
        "Features": best["Features"],
        "Test MAE": best["Test MAE"],
        "Test RMSE": best["Test RMSE"],
        "Test R²": best["Test R²"],
    })

comparison = pd.DataFrame(comparison_rows)
display(comparison.round(4))

In [ ]:
# Best descriptor and best fingerprint model under each protocol.
representation_rows = []

for split_name, table in result_tables.items():
    for features in ["Descriptors", "Fingerprints"]:
        subset = table[table["Features"] == features].copy()
        if len(subset) == 0:
            continue
        best = subset.sort_values("Test MAE").iloc[0]
        representation_rows.append({
            "Protocol": protocol_labels[split_name],
            "Features": features,
            "Best model": best["Model"],
            "Test MAE": best["Test MAE"],
            "Test RMSE": best["Test RMSE"],
            "Test R²": best["Test R²"],
        })

representation_comparison = pd.DataFrame(representation_rows)
display(representation_comparison.round(4))

In [ ]:
# Plot best MAE by split protocol.
plot_df = comparison.copy()

plt.figure(figsize=(9, 4.8))
plt.bar(
    plot_df["Protocol"],
    plot_df["Test MAE"],
    edgecolor="black"
)
plt.ylabel("Best Test MAE (kcal/mol)")
plt.xlabel("Evaluation protocol")
plt.title("Best classical model performance by split protocol")
plt.xticks(rotation=25, ha="right")
plt.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Plot descriptor vs fingerprint best model under each protocol.
pivot_mae = representation_comparison.pivot(
    index="Protocol",
    columns="Features",
    values="Test MAE"
).loc[comparison["Protocol"]]

ax = pivot_mae.plot(
    kind="bar",
    figsize=(9, 4.8),
    edgecolor="black"
)
ax.set_ylabel("Best Test MAE (kcal/mol)")
ax.set_xlabel("Evaluation protocol")
ax.set_title("Best descriptor model vs best fingerprint model")
ax.grid(True, axis="y", alpha=0.3)
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.show()

## Water Hold-Out Test

The water hold-out was performed in the original Notebook 2 as a solvent-specific external test using the descriptor Random Forest model selected from Section 4.

| Solvent | SMILES | n_train | n_test | R² | MAE | RMSE |
| --- | --- | --- | --- | --- | --- | --- |
| Water | O | 5597 | 642 | 0.0306 | 2.8091 | 3.7855 |
| DMSO | CS(C)=O | 6222 | 17 | 0.2500 | 1.0159 | 1.1254 |
| THF | C1CCOC1 | 6172 | 67 | 0.9488 | 0.3580 | 0.4640 |
| DCM | ClCCl | 6206 | 33 | 0.9303 | 0.3157 | 0.5928 |
| Octanol | CCCCCCCCO | 6110 | 129 | 0.9062 | 0.2902 | 0.7348 |
| Hexadecane | CCCCCCCCCCCCCCCC | 5960 | 279 | 0.9369 | 0.2573 | 0.3832 |
| Acetone | CC(C)=O | 6162 | 77 | 0.9652 | 0.2297 | 0.3476 |
| Ethanol | CCO | 6174 | 65 | 0.9835 | 0.2177 | 0.3287 |
| Toluene | Cc1ccccc1 | 6128 | 111 | 0.9761 | 0.2000 | 0.2922 |

### Is Water Hold-Out a Leave-Solvent-Out Test?

Yes. The water hold-out is a specific **leave-one-solvent-out** experiment: all water rows are removed from training and used as the test set, while the model trains on non-water solvents.

It is related to the new `leave_solvents_out` split, but it is not identical:

- `leave_solvents_out` is a general group-disjoint split where multiple solvent groups are assigned to train, validation, and test with no solvent overlap.
- The water hold-out deliberately chooses one chemically important solvent, water, as the held-out test domain.
- Because water is highly polar and hydrogen-bonding, it is a much harsher extrapolation test than many organic-solvent hold-outs.

This is why the water result is much worse than the general leave-solvents-out result: the model is not just seeing a new solvent label; it is being asked to extrapolate to a distinct aqueous-solvation regime.

In [ ]:
# Add water hold-out as a separate targeted protocol for visual comparison.
water_row = water_holdout[water_holdout["Solvent"] == "Water"].iloc[0]

comparison_with_water = pd.concat([
    comparison[["Protocol", "Train", "Validation", "Test", "Best model", "Features", "Test MAE", "Test RMSE", "Test R²"]],
    pd.DataFrame([{
        "Protocol": "Water-only hold-out",
        "Train": int(water_row["n_train"]),
        "Validation": np.nan,
        "Test": int(water_row["n_test"]),
        "Best model": "RandomForest",
        "Features": "Descriptors",
        "Test MAE": water_row["MAE"],
        "Test RMSE": water_row["RMSE"],
        "Test R²": water_row["R²"],
    }])
], ignore_index=True)

display(comparison_with_water.round(4))

In [ ]:
plt.figure(figsize=(10, 4.8))
plt.bar(
    comparison_with_water["Protocol"],
    comparison_with_water["Test MAE"],
    edgecolor="black"
)
plt.ylabel("Test MAE (kcal/mol)")
plt.xlabel("Evaluation protocol")
plt.title("Random/entity-disjoint protocols compared with water-only hold-out")
plt.xticks(rotation=25, ha="right")
plt.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

## Interpretation for the Dissertation

The fixed random row split remains the most optimistic benchmark because the same solutes and solvents can appear across train, validation, and test as different pairings. Its best model is descriptor SVR-rbf with Test MAE = 0.2113 kcal/mol.

The entity-disjoint splits reduce this overlap and therefore provide a more realistic test of generalisation. The best classical descriptor models still perform strongly: MAE = 0.3228 for scaffold-disjoint, 0.2534 for leave-solvents-out, and 0.2914 for leave-solutes-out. This supports the conclusion that RDKit pair descriptors contain robust predictive signal, not only row-level leakage.

However, the water-only hold-out is much harder, with MAE = 2.8091 kcal/mol and R² = 0.0306. This shows that performance can deteriorate sharply for a chemically distinctive solvent domain. Therefore, the dissertation should separate two claims: the models generalise reasonably under broad entity-disjoint splits, but aqueous-solvation extrapolation remains a difficult special case.

Across all protocols, descriptor-based models outperform Morgan-fingerprint models. This is consistent with the earlier Notebook 2 conclusion that continuous physicochemical descriptors are stronger than sparse binary fingerprints for this dataset.

## Recommended Reporting

For the main descriptor-vs-graph comparison, report the fixed random split alongside the three entity-disjoint splits. Use the fixed random split to compare with the original Notebook 3 ALIGNN/MGT runs, and use the entity-disjoint splits for the stricter leakage/generalisation analysis.

For the solvent-generalisation discussion, report the water hold-out separately as a targeted leave-one-solvent-out stress test rather than mixing it into the random group-disjoint split table.

In [ ]:
# Optional: export compact summary tables if this notebook is rerun.
from pathlib import Path

out_dir = Path("notebook2_final_descriptor_summary")
out_dir.mkdir(exist_ok=True)

comparison.to_csv(out_dir / "summary_best_by_protocol.csv", index=False)
representation_comparison.to_csv(out_dir / "summary_best_by_representation.csv", index=False)
water_holdout.to_csv(out_dir / "water_and_solvent_holdout_results.csv", index=False)
comparison_with_water.to_csv(out_dir / "summary_with_water_holdout.csv", index=False)

print(f"Saved compact summary CSVs to {out_dir}")